In [ ]:
# Purpose - plots fields and cuts for a single file

findex = 140 #140 #64 #67 #67 #67 #27 #25 #67 #26 #27 #2 #140 #143 #26 #63 #113 #121 #141 #130 #121

step_choice = "last" #"first" #all

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 1

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
from fmgmt.TristanRun import TristanRun
from fmgmt.FieldLineHelper import FieldLineHelper 

%aimport fmgmt.FieldLineHelper

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps[:-1], "penultimate": [run.steps[-2]]}

my_steps = tstep_choices[step_choice]
#my_steps = run.steps[-6:]

#boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
ts = 0

In [ ]:
# Plot Regions

buffer = 0
rect = [buffer, run.xlim - buffer, 0, run.ylim / 2]

roi = fq_cut(d.fields, ts, rect)

v_A = run.beta_A

plt.figure()

# --------------STREAMLINES--------------

#plot_streamline(plt, cs, cs.bx, cs.by, 3, 'white') 
#plot_streamline(plt, cs, cs.velx, cs.vely, 3, 'yellow') 

Az = compute_Az(roi)

Ainc = (0.16 * run.beta_A * run.cc) * run.B0 * run.output_interval

plt.contour(roi.x, roi.y, Az, np.linspace(np.min(Az), np.max(Az), 10) , colors = 'black', linewidths = 0.3, linestyles = 'solid') # '#00ff00')
plot_fq3(roi, Az)

plt.title(run.boilerplate + f"\n t = {run.steps[ts]} $\sigma =$ {run.sigma}")
#plt.vlines(ycut_x, rect[2], rect[3], colors = 'yellow', linestyle = '--')
#plt.hlines(xcut_y, rect[0], rect[1], colors = 'yellow', linestyle = '--')
plt.xlim(rect[:2])
plt.ylim(rect[2:])
plt.show()

In [ ]:
# Y-strip 

strip = ystrip

strip['ez_ideal'] = strip.bx * strip.vely - strip.by * strip.velx 
strip['ez_nonideal'] = strip.ez - strip.ez_ideal 
ez_tot = strip.ez

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez
dens = strip.dens1 + strip.dens2

esq, bsq, chi = sqrd(strip)

adj_factor = run.beta_A ** 2


bsq_adj = adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

sign_change = np.diff(np.sign(chi_adj))
#y_gate_index = np.where(sign_change > 0)[-1][-1]

jadj = 200

plt.figure()
plt.title(run.boilerplate + f"\n t = {run.steps[ts]}, x = {ycut_x}")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "ez_nontreal / c B_0")
#plt.plot(strip.y, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.y, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.y, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.y, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz
plt.plot(strip.y, strip.bx / B0, label = "$B_x / B_0$", color = '#ff7f0e')
#plt.plot(strip.y, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.y, chi, label = '$\chi$')
#plt.plot(strip.y, chi_adj, label = '$\chi_{adj}$', color = 'orange')
plt.plot(strip.y, strip.vely / run.beta_A, label = "$v_y / v_A$", color = '#9467bd')
#plt.plot(strip.y, strip.velx / run.beta_A, label = "$v_y / v_A$", color = 'red')
#plt.plot(strip.y, strip.dens / 16, label = '$n$')
#plt.plot(strip.y, (strip.vely / v_A) / (strip.bx / run.B0), label = "$v_y / v_A$ / $(B_x / B_0)$", color = '#8c564b')

#plt.plot(strip.y, strip.ez_nonideal / strip.jz, label = r"$\eta$", color = 'orange')


avg_velx_v_A = np.mean(strip.velx.values / run.beta_A)
print(avg_velx_v_A)

avg_velx_v_A = np.mean(dens.values / 16)
print(avg_velx_v_A)

plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = xcut_y, color = 'black', linestyle = '--')

#plt.plot(strip.y, jadj * ez_nontreal * strip.jz / (cc * B0 ** 2), label = f"{jadj} * ezni jz")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "$E_{zni}$ / $c B_0$", color = '#d62728')
#plt.axvline(x = strip.y[y_gate_index + 1], color = 'orange', linestyle = '--')

#plt.plot(strip.x, ez_nontreal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")

plt.legend(loc = 'lower right')
plt.xlabel('$y$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.xlim(rect[2:])
plt.show()

In [ ]:
print(0 / 0)

In [ ]:
print(0 / 0)

In [ ]:
print(run.steps)

In [ ]:
flh = FieldLineHelper(run, 16, contour_ds = 1)

rect = [55, 65, 110, 130]
rect = {10: [59.1, 60.9, 162, 167], 0.3: [59, 61, 105, 135]}[run.sigma]
rect = [55, 65, 155, 175]

counter = 0

for ts in range(len(my_steps)):
    plt.close('all')
    roi = fq_cut(d.fields, ts, rect)

    plt.figure()
    plt.title(f"{run.boilerplate} \n t = {run.steps[ts]}")
    
    plot_fq3(roi, roi.by, vabsmax = 5, cbar_label = '$B_y / B_0$', cmap = 'seismic')
    #plot_fq3(cs, cs.vely / v_A, vabsmax = 1.5, cmap = 'seismic', cbar_label = '$v_y / v_A$')
    #plot_streamline(plt, cs, cs.bx, cs.by, 1, 'black') 
    #plot_streamline(plt, cs, cs.velx, cs.vely, 0.5, 'black')
     
    # Downsample the grid for quiver
    step = 5
    xs = roi.x.values[::step]
    ys = roi.y.values[::step]
    X, Y = np.meshgrid(xs, ys)
    U = roi.velx.values[::step, ::step]
    V = roi.vely.values[::step, ::step]
    #plt.quiver(X, Y, U, V, color='white', scale=12, width=0.0055,     # Thinner arrows
    #       headlength=5,     # Optional: tweak arrowhead size
    #       headwidth=3)
    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'white') 
 
    plt.contour(roi.x, roi.y, roi.dens, [2.5], colors = 'green', label = 'n = 2.5')
    lim = 0.2
    plt.contour(roi.x, roi.y, roi.by, [-lim, lim], colors = 'black', label = f'B_y = +- {lim}')
    
    plt.xlim(rect[:2])
    plt.ylim(rect[2:])

    plt.legend()
    
    save_movie_frame(counter) 
    counter += 1

In [ ]:
counter = 0
#flh = FieldLineHelper(run, 8, contour_ds = 4)

j0 = cc * run.B0 / run.c_omp
rect = [40, 80, 100, 155]

for ts in range(len(my_steps))[:1]:#[:5]:
    
    plt.close('all')
    plt.figure()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
    plt.suptitle(f"{run.boilerplate} \n t = {run.steps[0] + ts}")
    roi = fq_cut(d.fields, ts, rect)

    # Left plot: current
    mesh1 = plot_fq3(roi, roi.bx / run.B0, plot=ax1, vabsmax=1.5, cmap='seismic')
    cbar1 = fig.colorbar(mesh1, ax=ax1)
    ax1.set_title('$B_x / B_0$')
    #flh.plot_field_lines(cs, cs.ez[0, 0], plot = ax1, arrow_spacing = -1, color = 'black')

    # Right plot: density
    mesh2 = plot_fq3(roi, roi.dens, cmap='viridis', plot=ax2, vabsmax=5, positive = True)
    cbar2 = fig.colorbar(mesh2, ax=ax2)
    ax2.set_title('$n / n_0$')
    #flh.plot_field_lines(cs, 0, plot = ax2, arrow_spacing = -1, color = 'white')

    plt.tight_layout()

    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'black') 

    for ax in [ax1, ax2]:
        ax.set_xlim(rect[:2])
        ax.set_ylim(rect[2:])
    
    save_movie_frame(counter) 
    counter += 1

    plt.show()

In [ ]:
counter = 0
#flh = FieldLineHelper(run, 8, contour_ds = 4)

j0 = cc * run.B0 / run.c_omp
rect = {26: [40, 80, 100, 155], 67: [50, 70, 145, 195]}[findex]

for ts in range(len(my_steps)):#[:5]:
    
    plt.close('all')
    plt.figure()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
    plt.suptitle(f"{run.boilerplate} \n t = {run.steps[0] + ts}")
    roi = fq_cut(d.fields, ts, rect)

    # Left plot: current
    mesh1 = plot_fq3(roi, roi.jz / j0, plot=ax1, vabsmax=1.5, cmap='seismic')
    cbar1 = fig.colorbar(mesh1, ax=ax1)
    ax1.set_title('$j_z / j_0$')
    #flh.plot_field_lines(cs, cs.ez[0, 0], plot = ax1, arrow_spacing = -1, color = 'black')

    # Right plot: density
    mesh2 = plot_fq3(roi, roi.dens, cmap='viridis', plot=ax2, vabsmax=5, positive = True)
    cbar2 = fig.colorbar(mesh2, ax=ax2)
    ax2.set_title('$n / n_0$')
    ax2.contour(roi.x, roi.y, roi.bx / run.B0, [-0.04, 0.04], colors = 'white')
    #flh.plot_field_lines(cs, 0, plot = ax2, arrow_spacing = -1, color = 'white')

    plt.tight_layout()

    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'black') 

    for ax in [ax1, ax2]:
        ax.set_xlim(rect[:2])
        ax.set_ylim(rect[2:])
    
    save_movie_frame(counter) 
    counter += 1

    #plt.show()

In [ ]:
print(sign_change_indices)

In [ ]:
# X-strip 

strip = xstrip

ez_ideal = strip.bx * strip.vely - strip.by * strip.velx 
ez_nonideal = strip.ez - ez_treal 
ez_tot = strip.ez
dens = strip.dens1 + strip.dens2

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez

esq, bsq, chi = sqrd(strip)


jadj = 200

plt.figure()
#plt.title(boilerplate + f"t = {tchosen}, y = {xcut_y}")

#plt.plot(strip.x, ez_nontreal / (cc * B0), label = "$E_{z, ni} / c B_0$", color = '#d62728')
#plt.plot(strip.x, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.x, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.x, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.x, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz

#plt.plot(strip.x, strip.by / B0, label = "$B_y / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, strip.epsilon, label = "$\epsilon$")
#plt.plot(strip.x, np.sqrt(bsq) / B0, label = "$B / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, bsq / B0 ** 2, label = "$B^2 / B_0^2$", color = '#ff7f0e')
#plt.plot(strip.x, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.x, chi, label = '$\chi$')
#plt.plot(strip.x, chi_adj, label = '$\chi_{adj}$')
#plt.plot(strip.x, strip.velx / v_A, label = "$v_x / v_A$", color = '#9467bd')
#plt.plot(strip.x, strip.vely / v_A, label = "$v_y / v_A$", color = 'magenta')
#plt.plot(strip.x, np.sqrt(strip.velx ** 2 + strip.vely ** 2) / v_A, label = "$v / v_A$", color = '#9467bd')
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = ycut_x, color = 'black', linestyle = '--')

#plt.plot(strip.x, strip.pb, label = "$p_B$")
#plt.plot(strip.x, strip.px, label = "$p_x$")

#plt.plot(strip.x, strip.pb + strip.px, label = "total pressure in x dir")

nbx = strip.bx / np.sqrt(bsq)
nby = strip.by / np.sqrt(bsq)
nbz = strip.bz / np.sqrt(bsq)
#ppara = strip.px * nbx + strip.py * nbx + strip.pz * nbz

#e_cr = (5 * strip.p + 2) / (5 * strip.p + 5)
#epsilon = 1 - 0.5 * (2 * ppara - strip.p)

#plt.plot(strip.x, e_cr, label = "$e_{cr}$")
#plt.plot(strip.x, epsilon, label = "$\epsilon$")

#plt.plot(strip.x, strip.px / strip.pb, label = "$beta$")

btot = np.sqrt(strip.bx ** 2 + strip.by ** 2 + strip.bz ** 2)

# New
#plt.plot(strip.x, strip.ex / (cc * B0), label = "$E_x / c B_0$")
#plt.plot(strip.x, strip.velz / v_A, label = "velz / v_A")
plt.plot(strip.x, strip.vely / v_A, label = "vely / v_A")
plt.plot(strip.x, strip.velx / v_A, label = "velx / v_A")
plt.plot(strip.x, strip.bx / run.B0, label = 'bx / B0')
plt.plot(strip.x, strip.by / run.B0, label = 'by / B0')
plt.plot(strip.x, btot / run.B0, label = 'btot')
plt.plot(strip.x, (strip.ez * strip.bx / btot ** 2) / run.beta_A , label =  'ecrossb over vA')
plt.plot(strip.x, strip.ez / run.B0, label = 'ez')
#plt.plot(strip.x, -strip.by * strip.velz / (cc * B0), label = "E_xni")
#plt.plot(strip.x, strip.dens1 / 16, label = 'n(e-)')
#plt.plot(strip.x, strip.dens2 / 16, label = 'n(e+)')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16, label = '$n$')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16 * (np.sqrt(bsq) / B0), label = '$(n / n_0) * (B / B_0)$', color = 'grey')
#plt.plot(strip.x, jadj * strip.jz / (dens * B0) * 16, label = f'{jadj} * j / (n / n0)')

#plt.plot(strip.x, (strip.dens2 - strip.dens1) / 16, label = 'rho')
#plt.plot(strip.x, jadj * strip.jx / B0, label = f'{jadj} * jx / B0')

#plt.plot(strip.x, ez_nonideal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title(boilerplate + f"x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.legend(loc='upper left')

plt.show()

In [ ]:
que = M_SM.values

In [ ]:
plt.figure()
plt.plot(strip.x, M_SM, label="M_SM")
plt.axhline(y=0, color='black', linestyle='--')
plt.axvline(x=ycut_x, color='black', linestyle='--')
plt.title(boilerplate + f" x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\\omega_{p}$]')
plt.ylabel('M_SM')
plt.legend(loc='center left')
plt.show()

In [ ]:
# Check if any element in in_sqrt is negative
negative_indexer = (in_sqrt < 0).compute()
negative_elements = in_sqrt[negative_indexer]

if negative_elements.size > 0:
    print("There are negative elements in in_sqrt.")
else:
    print("There are no negative elements in in_sqrt.")

In [ ]:
from scipy.integrate import simps

# Define the interval for integration
x_start, x_end = 332, 418
x_start, x_end = 340, 410


# Extract the relevant portion of the data
interval = (strip.x >= x_start) & (strip.x <= x_end)
x_values = strip.x[interval]
bsq_values = bsq[interval] / B0 ** 2

# Perform the integration
average_result = simps(bsq_values, x_values) / (x_end - x_start)

print(f"Integral of B^2 / B_0^2 over the interval [{x_start}, {x_end}] is {average_result}")

In [ ]:
print(list(strip.data_vars))

In [ ]:
print(y_gate[-1][-1])

In [ ]:
p